<a href="https://colab.research.google.com/github/prichter12/steam-steals/blob/main/steam_steals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steam Steals
My app helps customers decide what games to buy by determining what games have the highest values by their price.

In [8]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import requests
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded 1000 games from: /content/drive/MyDrive/Python/


In [9]:
from google.colab import userdata

steam_key = userdata.get("STEAM_API_KEY")

import requests

def get_owned_games(steam_id):
    url = "https://api.steampowered.com/IPlayerService/GetOwnedGames/v1/"

    params = {
        "key": steam_key,
        "steamid": steam_id,
        "include_appinfo": True,
        "format": "json"
    }

    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()

    return response.json()

In [10]:
steam_id = input("Enter your SteamID (17 digit number): ") #76561198287556178 is a test number

player_games = get_owned_games(steam_id)

owned_games = pd.DataFrame(player_games.get("response", {}).get("games", []))

if owned_games.empty:
    owned_games = pd.DataFrame(columns=["appid", "name", "playtime_forever"])

owned_games["playtime_hours"] = owned_games["playtime_forever"] / 60

total_games = len(owned_games)
total_hours = round(owned_games["playtime_hours"].sum(), 2)

player_history = owned_games.merge(
    games[["appid", "genres"]],
    on="appid",
    how="left"
)


Enter your SteamID (17 digit number): 76561198287556178


In [11]:
def time_value(games, n):
    paid = games[games["price"] > 0].copy()
    paid["hours_per_dollar"] = round(paid["avg_playtime_hours"] / paid["price"], 2)
    best = paid.sort_values("hours_per_dollar", ascending=False)
    return best[["name", "price", "hours_per_dollar"]].head(n)


def review_scores(games):
    paid = games[games["price"] > 0].copy()

    paid["review_score"] = paid["pct_pos_total"].rank(pct=True) * 100
    paid["price_score"] = paid["price"].rank(pct=True, ascending=False) * 100
    paid["reliability_score"] = paid["num_reviews_total"].rank(pct=True) * 100

    paid["value_score"] = (
        paid["review_score"] * 0.50 +
        paid["price_score"] * 0.30 +
        paid["reliability_score"] * 0.20
    )

    return paid


def price_reviews(games, n):
    paid = review_scores(games)
    best = paid.sort_values("value_score", ascending=False)
    return best[["name", "price", "value_score"]].head(n)


def community_value(games, n):
    paid = games[games["price"] > 0].copy()
    paid["users_per_dollar"] = round((paid["owners_low"] / paid["price"]), 2)
    best = paid.sort_values("users_per_dollar", ascending=False)
    return best[["name", "price", "users_per_dollar"]].head(n)

In [12]:
def favorite_genres(player_history):
    history = player_history.dropna(subset=["genres"]).copy()
    history["genres"] = history["genres"].str.split(",")
    history = history.explode("genres")
    history["genres"] = history["genres"].str.strip()

    favorites = history.groupby("genres")["playtime_hours"].sum()
    return favorites.sort_values(ascending=False)


def recommend_games(games, owned_games, player_history, n):
    favorites = favorite_genres(player_history).head(3).index.tolist()

    available = overall_scores(games)

    available = available[~available["appid"].isin(owned_games["appid"])].copy()

    if favorites:
        available = available[
            available["genres"].fillna("").apply(
                lambda x: any(genre.strip() in favorites for genre in x.split(","))
            )
        ]

    best = available.sort_values("overall_score", ascending=False)

    return best[["name", "genres", "price", "overall_score"]].head(n)

In [14]:
print("STEAMSTEALS RESULTS")
print("------------------------")

print("Games owned:", total_games)
print("Total hours played:", total_hours)

favorites = favorite_genres(player_history)

if not favorites.empty:
    print("Favorite genre:", favorites.index[0])
else:
    print("Favorite genre: Not enough data")

print("\nTOP 5 PERSONALIZED RECOMMENDATIONS")

recommendations = recommend_games(games, owned_games, player_history, 5)

print(recommendations.round(2))

STEAMSTEALS RESULTS
------------------------
Games owned: 21
Total hours played: 4993.05
Favorite genre: Strategy

TOP 5 PERSONALIZED RECOMMENDATIONS
                 name                                  genres  price  \
63  Vampire Survivors              Action, Casual, Indie, RPG   4.99   
54  People Playground       Action, Casual, Indie, Simulation   9.99   
6            Terraria           Action, Adventure, Indie, RPG   9.99   
32     Lethal Company  Action, Adventure, Indie, Early Access   9.99   
13     Stardew Valley                  Indie, RPG, Simulation  14.99   

    value_score  
63        96.20  
54        92.65  
6         90.84  
32        90.40  
13        90.20  
